# 10 — ML Classification

Trains and evaluates zone-type classification models on the combined feature dataset.

**Models:** Logistic Regression (baseline), Random Forest, XGBoost

**Evaluation:**
- Spatial cross-validation (GroupKFold by tract geography)
- Class-weighted training to handle imbalance
- Per-class F1, macro-F1, confusion matrix
- Feature importance analysis

**Output:** Classification plots to `outputs/latest/`

In [ ]:
CSV_PATH = ""  # auto-detect if empty
PLOTS_DIR = "outputs/latest"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)

os.makedirs(PLOTS_DIR, exist_ok=True)

# Auto-detect latest combined CSV
if not CSV_PATH:
    csvs = sorted(glob.glob("csv/combined_zones_*.csv"))
    if not csvs:
        raise FileNotFoundError("No combined_zones CSV found in csv/")
    CSV_PATH = csvs[-1]

df = pd.read_csv(CSV_PATH, dtype={"tract_id": str})
print(f"Loaded: {CSV_PATH}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ── Data inspection ───────────────────────────────────

print("Zone type distribution:")
print(df["zone_type"].value_counts().to_string())

print(f"\nColumn completeness:")
for col in df.columns:
    n = df[col].notna().sum()
    pct = 100 * n / len(df)
    print(f"  {col:<30s} {n:>4d}/{len(df)}  ({pct:.1f}%)")

print(f"\nData types:")
print(df.dtypes.to_string())

In [ ]:
# ── Prepare features and target ───────────────────────

# Columns to exclude from features
EXCLUDE = ["tract_id", "borough", "tract_lat", "tract_lon", "zone_type", "tract_lot_count"]

feature_cols = [c for c in df.columns if c not in EXCLUDE]
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

# Drop rows where target is missing
df = df.dropna(subset=["zone_type"]).copy()

# Drop columns that are 100% null
null_cols = [c for c in feature_cols if df[c].isna().all()]
if null_cols:
    print(f"Dropping 100% null columns: {null_cols}")
    feature_cols = [c for c in feature_cols if c not in null_cols]

# Impute remaining nulls with median
for col in feature_cols:
    if df[col].isna().any():
        median_val = df[col].median()
        n_missing = df[col].isna().sum()
        df[col] = df[col].fillna(median_val)
        print(f"  Imputed {col}: {n_missing} nulls → median {median_val:.2f}")

X = df[feature_cols].values
le = LabelEncoder()
y = le.fit_transform(df["zone_type"])
class_names = le.classes_

print(f"\nX shape: {X.shape}")
print(f"Classes: {list(class_names)}")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
# ── Zone type distribution plot ───────────────────────

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=df, y="zone_type", order=df["zone_type"].value_counts().index,
              palette="viridis", ax=ax)
ax.set_title("Zone Type Distribution")
ax.set_xlabel("Count")
ax.set_ylabel("Zone Type")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/zone_type_distribution.png", dpi=150)
plt.show()

In [ ]:
# ── Feature correlation ───────────────────────────────

fig, ax = plt.subplots(figsize=(14, 10))
corr = df[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, vmin=-1, vmax=1, square=True,
            xticklabels=[c.replace('_', '\n') for c in feature_cols],
            yticklabels=[c.replace('_', '\n') for c in feature_cols])
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/feature_correlation.png", dpi=150)
plt.show()

In [ ]:
# ── Spatial Cross-Validation setup ────────────────────
# Group tracts into spatial blocks using lat bins for GroupKFold
# This prevents spatial leakage (nearby tracts in both train and test)

N_FOLDS = 5

# Create spatial groups by binning latitude into N_FOLDS bins
lat_bins = pd.qcut(df["tract_lat"], q=N_FOLDS, labels=False)
groups = lat_bins.values

gkf = GroupKFold(n_splits=N_FOLDS)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Spatial CV: {N_FOLDS} folds")
print(f"Samples per fold:")
for fold_i, (train_idx, test_idx) in enumerate(gkf.split(X_scaled, y, groups)):
    print(f"  Fold {fold_i+1}: train={len(train_idx)}, test={len(test_idx)}")

In [ ]:
# ── Train models ──────────────────────────────────────

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42, multi_class="multinomial"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1
    ),
}

# Try XGBoost if available
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, use_label_encoder=False, eval_metric="mlogloss",
        verbosity=0
    )
except ImportError:
    print("XGBoost not installed — skipping.")

results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    
    # Cross-validated predictions
    y_pred = cross_val_predict(model, X_scaled, y, groups=groups, cv=gkf)
    
    acc = accuracy_score(y, y_pred)
    f1_macro = f1_score(y, y_pred, average="macro")
    f1_weighted = f1_score(y, y_pred, average="weighted")
    
    print(f"  Accuracy:    {acc:.3f}")
    print(f"  Macro F1:    {f1_macro:.3f}")
    print(f"  Weighted F1: {f1_weighted:.3f}")
    print()
    print(classification_report(y, y_pred, target_names=class_names))
    
    results[name] = {
        "y_pred": y_pred,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }

In [ ]:
# ── Confusion matrices ────────────────────────────────

n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6))
if n_models == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y, res["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{name}\nMacro F1: {res['f1_macro']:.3f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
# ── Feature importance (Random Forest) ────────────────

# Retrain on full data for feature importance
rf = RandomForestClassifier(
    n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_scaled, y)

importances = rf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, max(6, len(feature_cols) * 0.4)))
sns.barplot(
    x=importances[sorted_idx],
    y=[feature_cols[i] for i in sorted_idx],
    palette="viridis", ax=ax
)
ax.set_title("Feature Importance (Random Forest)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/feature_importance.png", dpi=150)
plt.show()

print("Feature importance ranking:")
for rank, idx in enumerate(sorted_idx, 1):
    print(f"  {rank:>2d}. {feature_cols[idx]:<30s} {importances[idx]:.4f}")

In [ ]:
# ── Model comparison summary ──────────────────────────

print("\n" + "=" * 60)
print("  MODEL COMPARISON")
print("=" * 60)
print(f"  {'Model':<25s} {'Accuracy':>10s} {'Macro F1':>10s} {'Weighted F1':>12s}")
print(f"  {'-'*25} {'-'*10} {'-'*10} {'-'*12}")

for name, res in results.items():
    print(f"  {name:<25s} {res['accuracy']:>10.3f} {res['f1_macro']:>10.3f} {res['f1_weighted']:>12.3f}")

best_model = max(results, key=lambda k: results[k]["f1_macro"])
print(f"\n  Best model (macro F1): {best_model}")

print(f"\nPlots saved to: {PLOTS_DIR}/")
for f in sorted(os.listdir(PLOTS_DIR)):
    if f.endswith(".png"):
        print(f"  {f}")